# Demos: Lecture 11 

## Hands-on with quantum compilation

On Tuesday we saw examples of circuit rewrite rules and optimizations, such as converting $HZH$ into $X$, or substituting $Z$ for $TTTT$. Manually applying such optimizations would be really tedious. Thankfully, PennyLane has mechanisms available that make it easier to perform repeated and complex modifications to circuits using *quantum transforms*. 

In these demos, we will
 - apply transforms to optimize quantum circuits in PennyLane
 - compile Grover's algorithm into 1- and 2-qubit gates to determine how much it *really* costs
 - describe the process of qubit routing, and compute overheads for the process

In [ ]:
import pennylane as qml
from pennylane import numpy as np
from functools import partial
import matplotlib.pyplot as plt

qml.decomposition.enable_graph()

In [ ]:
def construct_grover_qnode(special_string, num_steps):
    n_bits = len(special_string)
    n_work_wires = 1
    
    def hadamard_transform(wires):
        for wire in wires:
            qml.Hadamard(wires=wire)
    
    def oracle():
        qml.MultiControlledX(
            wires=range(n_bits+1), 
            control_values=special_string,
            work_wires=range(n_bits+1, n_bits+1+n_work_wires)
        )
    
    def diffusion():
        hadamard_transform(wires=range(n_bits))
        qml.MultiControlledX(
            wires=range(n_bits+1),
            control_values=[0]*n_bits,
            work_wires=range(n_bits+1, n_bits+1+n_work_wires)
        )
        hadamard_transform(wires=range(n_bits))
    
    dev = qml.device('default.qubit', wires=n_bits+1+n_work_wires)

    def grover():
        qml.PauliX(wires=n_bits)
        hadamard_transform(wires=range(n_bits+1))
            
        for _ in range(num_steps):
            oracle()
            diffusion()
            
        return qml.probs(wires=range(n_bits))    

    return qml.QNode(grover, dev)

In [ ]:
special_string = [1, 1, 0, 1]
num_steps = 1

grover_qnode = construct_grover_qnode(special_string, num_steps)

In [ ]:
print(qml.draw(grover_qnode)())

In [ ]:
print(qml.specs(grover_qnode)()['resources'])

In [ ]:
special_string_lengths = list(range(2, 8))

depths = []
gate_counts = []

for special_length in special_string_lengths:
    special_string = np.random.randint(2, size=special_length).tolist()
    num_steps = 1
    
    grover_qnode = construct_grover_qnode(special_string, num_steps)

    resources = qml.specs(grover_qnode)()['resources']
    depths.append(resources.depth)
    gate_counts.append(resources.num_gates)

plt.scatter(special_string_lengths, depths, label="Depth")
plt.scatter(special_string_lengths, gate_counts, label="Num. gates")
plt.legend()

## Demo 2: Decomposition

In [ ]:
#gate_set={"Toffoli", "CNOT", "X",  "Hadamard", "T", "Adjoint(T)"}
gate_set={"CNOT", "X",  "Hadamard", "T", "Adjoint(T)"}

special_string = np.random.randint(2, size=3).tolist()
num_steps = 1
grover_qnode = construct_grover_qnode(special_string, num_steps)

decomposed_grover_qnode = qml.transforms.decompose(grover_qnode, gate_set=gate_set)

In [ ]:
print(qml.draw(decomposed_grover_qnode)())

In [ ]:
print(qml.specs(decomposed_grover_qnode)()['resources'])

In [ ]:
decomposed_depths = []
decomposed_gate_counts_1 = []
decomposed_gate_counts_2 = []

for special_length in special_string_lengths:
    special_string = np.random.randint(2, size=special_length).tolist()
    num_steps = 1
    
    grover_qnode = construct_grover_qnode(special_string, num_steps)

    decomposed_grover_qnode = qml.transforms.decompose(grover_qnode, gate_set=gate_set)

    resources = qml.specs(decomposed_grover_qnode)()['resources']
    decomposed_depths.append(resources.depth)
    decomposed_gate_counts_1.append(resources.gate_sizes[1])
    decomposed_gate_counts_2.append(resources.gate_sizes[2])

plt.scatter(special_string_lengths, decomposed_depths, label="Depth")
plt.scatter(special_string_lengths, decomposed_gate_counts_1, label="1-qubit gates")
plt.scatter(special_string_lengths, decomposed_gate_counts_2, label="2-qubit gates")
plt.legend()

## Demo 3: Optimization

In [ ]:
s_to_z = [qml.S(0), qml.S(0), qml.Z(0)]
t_to_s = [qml.T(0), qml.T(0), qml.adjoint(qml.S)(0)]
t_to_z = [qml.T(0), qml.T(0), qml.T(0), qml.T(0), qml.Z(0)]
pattern_tapes = [qml.tape.QuantumTape(tape) for tape in [t_to_z, t_to_s, s_to_z]]

pipeline = [
    partial(qml.transforms.decompose, gate_set=gate_set),
    qml.transforms.cancel_inverses,
    qml.transforms.commute_controlled,
    partial(qml.transforms.pattern_matching_optimization, pattern_tapes=pattern_tapes)
]

special_string = np.random.randint(2, size=3).tolist()
num_steps = 1

grover_qnode = construct_grover_qnode(special_string, num_steps)
optimized_grover_qnode = qml.compile(grover_qnode, pipeline=pipeline)

In [ ]:
print(qml.draw(optimized_grover_qnode)())

In [ ]:
print(qml.specs(optimized_grover_qnode)()['resources'])

In [ ]:
optimized_depths = []
optimized_gate_counts_1 = []
optimized_gate_counts_2 = []

for special_length in special_string_lengths:
    special_string = [1] + [0] * (special_length - 1)
    num_steps = 1
    
    grover_qnode = construct_grover_qnode(special_string, num_steps)

    optimized_grover_qnode = qml.compile(grover_qnode, pipeline=pipeline)

    resources = qml.specs(optimized_grover_qnode)()['resources']
    optimized_depths.append(resources.depth)
    optimized_gate_counts_1.append(resources.gate_sizes[1])
    optimized_gate_counts_2.append(resources.gate_sizes[2])

plt.scatter(special_string_lengths, decomposed_depths, label="Depth")
plt.scatter(special_string_lengths, decomposed_gate_counts_1, label="1-qubit gates")
plt.scatter(special_string_lengths, decomposed_gate_counts_2, label="2-qubit gates")

plt.scatter(special_string_lengths, optimized_depths, color='tab:blue', marker="v")
plt.scatter(special_string_lengths, optimized_gate_counts_1, color='tab:orange', marker="v")
plt.scatter(special_string_lengths, optimized_gate_counts_2, color='tab:green', marker="v")

plt.xlabel("String length")
plt.xticks(special_string_lengths)
plt.legend()

## Demo 4: topology-restricted processor

<img src="fig/processor-topology.png" width="400px">


In [ ]:
special_string = [1, 0, 0, 0]

grover_qnode = construct_grover_qnode(special_string, num_steps)
optimized_grover_qnode = qml.compile(grover_qnode, pipeline=pipeline)

coupling_map = [(i, i+1) for i in range(len(special_string) + 1)]

transpiled_grover_qnode = qml.transforms.transpile(optimized_grover_qnode, coupling_map=coupling_map)

In [ ]:
print(qml.specs(transpiled_grover_qnode)()['resources'])

In [ ]:
print(qml.draw(transpiled_grover_qnode)())

In [ ]:
depth_ratio = []

for special_length in special_string_lengths:
    special_string = [1] + [0] * (special_length - 1)
    num_steps = 1
    
    grover_qnode = construct_grover_qnode(special_string, num_steps)
    optimized_grover_qnode = qml.compile(grover_qnode, pipeline=pipeline)

    coupling_map = [(i, i+1) for i in range(len(optimized_grover_qnode.device.wires) - 1)]
    transpiled_grover_qnode = qml.transforms.transpile(optimized_grover_qnode, coupling_map=coupling_map)

    resources_optimized = qml.specs(optimized_grover_qnode)()['resources']
    resources_routed = qml.specs(transpiled_grover_qnode)()['resources']
    depth_ratio.append(resources_routed.depth / resources_optimized.depth)
    
plt.scatter(special_string_lengths, depth_ratio)
plt.xlabel("String length")
plt.xticks(special_string_lengths)
plt.ylabel("Depth ratio")